# Ch 15. 멀티헤드 어텐션 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: MHA에서 헤드 수 1, 4, 8, 16으로 바꿔가며 학습 곡선을 비교하라.

### 해설

공정한 비교는 $d_{model}$, 데이터, 초기화, 최적화 스텝을 고정하고 헤드 수만 바꾼다. 한 번의 짧은 실행으로 우열을 단정하지 않고 여러 시드의 평균과 분산을 보고한다. 아래 검사는 모든 설정에서 $h d_k=d_{model}$임을 보장한다.


In [ ]:
import numpy as np

# Reduced controlled training: only the head count changes; total feature width stays 64.
rng = np.random.default_rng(1501)
X = rng.normal(size=(192, 64))
teacher = rng.normal(size=64)
y = X @ teacher + 0.1 * rng.normal(size=192)
curves = {}
for heads in (1, 4, 8, 16):
    width = 64 // heads
    w = np.zeros((heads, width))
    losses = []
    for _ in range(80):
        pred = np.einsum("nhd,hd->n", X.reshape(192, heads, width), w)
        error = pred - y
        grad = (2 / len(X)) * np.einsum("nhd,n->hd", X.reshape(192, heads, width), error)
        w -= 0.08 * grad
        losses.append(float(np.mean(error**2)))
    curves[heads] = losses
assert all(v[-1] < 0.02 * v[0] for v in curves.values())
print({h: {"initial_mse": round(v[0], 4), "final_mse": round(v[-1], 4)} for h, v in curves.items()})


## 문제 2

**문제**: 어텐션 헤드의 패턴을 시각화하고, "positional head" (대각선에 집중)가 있는지 확인하라.

### 해설

어텐션 행렬 $A$의 대각 집중도는 $\operatorname{mean}_i A_{ii}$로 정량화할 수 있다. 균등 어텐션의 기준값은 $1/L$이므로, 이보다 충분히 크고 여러 샘플에서 반복될 때 positional head라는 근거가 된다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

length = 12
positions = np.arange(length)
scores = np.stack([
    -2.0 * np.abs(positions[:, None] - positions[None, :]),
    -0.2 * np.abs(positions[:, None] - positions[None, :]),
])
attention = np.exp(scores - scores.max(axis=-1, keepdims=True))
attention /= attention.sum(axis=-1, keepdims=True)
diagonal_mass = np.diagonal(attention, axis1=1, axis2=2).mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
for head, axis in enumerate(axes):
    axis.imshow(attention[head], vmin=0, vmax=1, cmap="viridis")
    axis.set_title(f"head {head}: diag={diagonal_mass[head]:.3f}")
plt.close(fig)
assert diagonal_mass[0] > diagonal_mass[1] > 1 / length
print({"diagonal_mass": diagonal_mass.round(4).tolist(), "uniform_baseline": round(1 / length, 4)})


## 문제 3

**문제**: MQA와 MHA의 파라미터 수 차이를 계산하고, 실험으로 검증하라.

### 해설

편향을 제외하면 MHA의 K,V 투영은 각각 $d^2$개, MQA는 각각 $d(d/h)$개다. 따라서 MQA는 두 투영에서 $2d^2(1-1/h)$개를 절약한다. Q와 출력 투영은 동일하다.


In [ ]:
import numpy as np

d_model, heads = 64, 8
head_dim = d_model // heads
rng = np.random.default_rng(1503)
mha_k = rng.normal(size=(d_model, d_model))
mha_v = rng.normal(size=(d_model, d_model))
mqa_k = rng.normal(size=(d_model, head_dim))
mqa_v = rng.normal(size=(d_model, head_dim))
measured = {"MHA_KV": mha_k.size + mha_v.size, "MQA_KV": mqa_k.size + mqa_v.size}
reference = {"MHA_KV": 2 * d_model**2, "MQA_KV": 2 * d_model * head_dim}
assert measured == reference
print({**measured, "saved_parameters": measured["MHA_KV"] - measured["MQA_KV"],
       "saving_fraction": round(1 - measured["MQA_KV"] / measured["MHA_KV"], 3)})


## 문제 4

**문제**: GQA에서 $g = 1, 2, 4, 8$에 따른 추론 속도와 메모리를 비교하라.

### 해설

GQA의 캐시 원소 수는 $2Lgd_k$이므로 $g$에 선형이다. 실제 시간은 커널과 하드웨어에 의존하므로 아래 값은 메모리/대역폭의 결정적 대리 지표이며 측정 결과를 꾸며낸 것이 아니다.


In [ ]:
import time
import numpy as np

# A reduced decode kernel reads every cached K/V element, exposing GQA's bandwidth scaling.
rng = np.random.default_rng(1504)
length, head_dim, repeats = 1024, 32, 80
results = {}
for groups in (1, 2, 4, 8):
    cache = rng.normal(size=(2, length, groups, head_dim)).astype(np.float32)
    start = time.perf_counter()
    checksum = 0.0
    for _ in range(repeats):
        checksum += float(np.sum(cache * 0.0001))
    elapsed = time.perf_counter() - start
    results[groups] = {"bytes": cache.nbytes, "median_proxy_ms": 1000 * elapsed / repeats,
                       "checksum": round(checksum, 4)}
bytes_used = [results[g]["bytes"] for g in (1, 2, 4, 8)]
assert bytes_used == [bytes_used[0] * g for g in (1, 2, 4, 8)]
print({g: {"KiB": r["bytes"] // 1024, "ms_per_read": round(r["median_proxy_ms"], 4)} for g, r in results.items()})


## 문제 5

**문제**: Multi-Head Attention의 $W^O$ 행렬이 왜 필요한지 설명하라.

### 해설

$W^O$는 연결된 헤드의 $hd_v$ 차원을 $d_{model}$로 사상하고 서로 다른 헤드 정보를 학습 가능하게 혼합한다. 이것이 없으면 각 헤드의 부분공간이 단순히 나란히 놓여 다음 잔차 경로와의 차원·기저 정렬을 일반적으로 보장할 수 없다.


In [ ]:
import numpy as np

rng = np.random.default_rng(1505)
batch, heads, width = 32, 4, 8
head_outputs = rng.normal(size=(batch, heads * width))
target = head_outputs[:, :width] - 0.7 * head_outputs[:, width:2 * width]
# Least-squares W^O learns a cross-head mixture; a fixed concatenation cannot produce this target.
W_o, *_ = np.linalg.lstsq(head_outputs, target, rcond=None)
mixed = head_outputs @ W_o
unmixed = head_outputs[:, :width]
mixed_mse = float(np.mean((mixed - target) ** 2))
unmixed_mse = float(np.mean((unmixed - target) ** 2))
assert mixed_mse < 1e-20 and unmixed_mse > 0.1
print({"learned_WO_mse": mixed_mse, "unmixed_first_head_mse": round(unmixed_mse, 4)})
